# Test the AITA verdict classifier

Loads the fine-tuned `roberta-base` model and predicts **NTA** (not the a-hole) vs **YTA** (you're the a-hole) for any post text.

**Before you run:**
1. Select the project's **`.venv`** (Python 3.12) as the kernel — it has `torch` / `transformers`. In VS Code: pick the kernel in the top-right and choose `.venv`.
2. The **`out/model/`** folder must exist. Either train it (`python model/train.py ...`) or unzip the downloaded model into `out/model/`.
3. Run the cells top to bottom. Paths are relative to the repo root (where this notebook lives).

In [ ]:
import os, json, torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification

# Folder with the saved fine-tuned model (train.py output, or unzip here).
MODEL_DIR = "out/model"
ID2LABEL = {0: "NTA", 1: "YTA"}

assert os.path.isdir(MODEL_DIR), f"Model not found at {MODEL_DIR!r} - train it or unzip the download there."
device = "cuda" if torch.cuda.is_available() else "cpu"
tok = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).to(device).eval()
print("Model loaded on", device)

In [ ]:
@torch.no_grad()
def predict(text, max_len=256):
    enc = tok(text, truncation=True, max_length=max_len, return_tensors="pt").to(device)
    probs = torch.softmax(model(**enc).logits, dim=-1)[0].tolist()
    pred = max(range(len(probs)), key=lambda i: probs[i])
    return ID2LABEL[pred], {ID2LABEL[i]: round(p, 4) for i, p in enumerate(probs)}

def show(text):
    label, probs = predict(text)
    meaning = "you are NOT the asshole" if label == "NTA" else "YOU are the asshole"
    print(text.strip()[:220])
    print(f"  -> {label}  ({meaning})   {probs}")
    print("  (reflects how r/AmItheAsshole would likely vote, not an objective judgement)\n")

## A few examples

In [ ]:
examples = [
    "AITA for refusing to give up my paid-for window seat so a family could sit together?",
    "AITA for eating the lunch my roommate clearly labeled with her name in the fridge?",
    "AITA for reporting my coworker to HR for taking credit for a project I did by myself?",
    "WIBTA if I skipped my best friend's destination wedding because I can't afford the trip?",
]
for e in examples:
    show(e)

## Try your own
Edit `my_post` below and re-run the cell.

In [ ]:
my_post = """
AITA for telling my coworker to stop microwaving fish in the shared office kitchen?
"""

show(my_post)

## Batch test on the hand-built evaluation cases
Runs all 13 labeled cases and reports accuracy.

In [ ]:
cases = [json.loads(l) for l in open("model/eval_cases.jsonl", encoding="utf-8") if l.strip()]
correct = 0
for c in cases:
    label, _ = predict(c["text"])
    ok = label == c["expected"]
    correct += ok
    print(f"[{'OK' if ok else 'XX'}] pred={label} exp={c['expected']}  {c['text'][:62]}...")
print(f"\n{correct}/{len(cases)} correct ({correct/len(cases):.0%})")